In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/CryAndRRich/codapath.git"
REPO_BRANCH = "namhai"
REPO = Path("/kaggle/working/codapath")

if (REPO / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "switch", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH])
elif REPO.exists():
    raise RuntimeError(f"{REPO} exists but is not a Git repository")
else:
    subprocess.check_call(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO)]
    )

branch = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "--abbrev-ref", "HEAD"], text=True
).strip()
commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], text=True
).strip()
print(f"codapath @ {branch} {commit}")


In [ ]:
%cd /kaggle/working/codapath


In [ ]:
import os
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U",
                       "huggingface_hub<1.0", "hf-transfer"])

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
if "/kaggle/working/codapath" not in sys.path:
    sys.path.append("/kaggle/working/codapath")


In [ ]:
import glob
import statistics

import numpy as np
import torch

from data.loaders import get_data_loaders
from evaluation import (
    find_run_dir,
    load_test_features,
    make_lora_encoder,
    probe_feature_paths,
    read_run_metadata,
    rescore_run,
)
from utils.kaggle import find_data_root


In [ ]:
WEIGHTS_ROOT = "/kaggle/input/EDIT_WEIGHTS_SLUG"   # holds PACT/ and/or baselines/
FEATURE_ROOT = "/kaggle/input/EDIT_FEATURES_SLUG"  # holds DINO_embed/ and/or CONCH_embed/

DATASET = "histoset"   # pathmnist | histoset | skintissue
SEEDS = [38, 42, 611]  # every seed to score

METRICS = ["acc", "precision", "recall", "f1"]  # any subset, in report order

# Which run directories to score. "" scores every run found under both
# WEIGHTS_ROOT/PACT and WEIGHTS_ROOT/baselines for this dataset and seed.
RUN_FILTER = ""  # substring of the run directory name, e.g. "poolcons5" or "random"

SCORE_LORA_RUNS = False  # True re-encodes the test set per budget: GPU + hours


In [ ]:
# Every number below is COMPUTED here from the saved weights. The weight
# archives contain nothing else: no `_results.pt`, no predictions, no logs --
# so there is no recorded metric to fall back on, by construction. What each
# run was (its encoder, dataset, seed) is read from the `metadata` dict inside
# its own probe checkpoint.
run_dirs = []
for family in ("PACT", "baselines"):
    for seed in SEEDS:
        pattern = os.path.join(WEIGHTS_ROOT, family, DATASET, f"seed{seed}", "*")
        for path in sorted(glob.glob(pattern)):
            if not os.path.isdir(path):
                continue
            if RUN_FILTER and RUN_FILTER not in os.path.basename(path):
                continue
            run_dirs.append((family, seed, path))

assert run_dirs, (
    f"no run directories under {WEIGHTS_ROOT}/{{PACT,baselines}}/{DATASET}/seed* "
    f"matching {RUN_FILTER!r}"
)
print(f"{len(run_dirs)} run directories | dataset={DATASET} | seeds={SEEDS}")


In [ ]:
# Test LABELS come from the image dataset, not from any archive: a probe is
# scored against the split it was trained against, and only the dataset knows
# it. The loader is the same fixed-order one every run used, so row i of the
# feature cache is row i here.
DATA_ROOT = find_data_root()
DATA_PATHS = {
    "pathmnist": str(DATA_ROOT / "pathmnist_224.npz"),
    "histoset": str(DATA_ROOT / "HistoSet-5x14/HistoSet-5x14"),
    "skintissue": str(DATA_ROOT / "SkinTissue/SkinTissue/tiles"),
}
data_path = DATA_PATHS[DATASET]
assert Path(data_path).exists(), f"Missing Kaggle input: {data_path}"

test_labels_by_seed = {}
test_dataset_by_seed = {}
for seed in SEEDS:
    _, test_loader, class_names = get_data_loaders(data_path, seed=seed)
    labels = np.concatenate([batch[1].numpy() for batch in test_loader])
    test_labels_by_seed[seed] = labels
    # The dataset itself is kept for the LoRA path, which must re-encode these
    # exact rows in this exact order through the adapted encoder.
    test_dataset_by_seed[seed] = test_loader.dataset
    print(f"seed {seed}: {len(labels)} test rows, {len(class_names)} classes")


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

scored = {}   # (run_name, seed) -> {budget: {metric: value}}
skipped = []

for family, seed, run_dir in run_dirs:
    info = read_run_metadata(run_dir)
    run_name = info["run_name"]
    assert info["dataset"] == DATASET, f"{run_dir}: metadata says {info['dataset']}"
    assert info["seed"] == seed, f"{run_dir}: metadata says seed {info['seed']}"

    if info["has_lora"] and not SCORE_LORA_RUNS:
        skipped.append((run_name, seed, "LoRA run: set SCORE_LORA_RUNS=True"))
        continue

    if info["has_lora"]:
        # The frozen cache is the WRONG SPACE for this run, so it is not even
        # opened: the adapter re-encodes the test set per budget instead.
        features = None
        encode_test = make_lora_encoder(info, test_dataset_by_seed[seed], device)
    else:
        is_conch = info["encoder_kind"] == "conch"
        cache_root = os.path.join(
            FEATURE_ROOT, "CONCH_embed" if is_conch else "DINO_embed"
        )
        paths = probe_feature_paths(cache_root, DATASET, seed, info["encoder"])
        features = load_test_features(paths, expect_conch=is_conch)
        encode_test = None

    scored[(run_name, seed)] = rescore_run(
        run_dir, features, test_labels_by_seed[seed], device, encode_test=encode_test
    )
    print(f"scored {run_name} seed={seed} "
          f"({len(scored[(run_name, seed)])} budgets, {info['encoder_kind']})")

for run_name, seed, why in skipped:
    print(f"skipped {run_name} seed={seed}: {why}")


In [ ]:
budgets = sorted({b for run in scored.values() for b in run})

for metric in METRICS:
    print(f"=== {metric} (%) -- computed from the saved weights ===")
    header = f"{'run':<44}{'seed':>5}" + "".join(f"{b:>8}" for b in budgets) + f"{'mean':>9}"
    print(header)
    for (run_name, seed) in sorted(scored):
        row = scored[(run_name, seed)]
        values = [row[b][metric] * 100.0 for b in budgets if b in row]
        cells = "".join(f"{row[b][metric] * 100:>8.2f}" if b in row else f"{'-':>8}"
                        for b in budgets)
        mean = statistics.mean(values) if values else float("nan")
        print(f"{run_name[:43]:<44}{seed:>5}{cells}{mean:>9.2f}")
    print()


In [ ]:
# Mean +- std over seeds, per run name.
by_run = {}
for (run_name, seed), row in scored.items():
    by_run.setdefault(run_name, {})[seed] = row

print(f"{'run':<44}{'budget':>8}{'mean':>9}{'std':>9}{'seeds':>7}")
for run_name in sorted(by_run):
    per_seed = by_run[run_name]
    for budget in budgets:
        values = [per_seed[s][budget]["acc"] * 100.0
                  for s in sorted(per_seed) if budget in per_seed[s]]
        if not values:
            continue
        std = statistics.stdev(values) if len(values) > 1 else 0.0
        print(f"{run_name[:43]:<44}{budget:>8}{statistics.mean(values):>9.2f}"
              f"{std:>9.2f}{len(values):>7}")


In [ ]:
# A sanity check the weights can answer on their own: every probe in a run
# must have the same shape and the same class count, and accuracy must rise
# with budget more often than it falls. A run that fails this is not reporting
# a worse method -- it is reporting a broken archive.
for (run_name, seed) in sorted(scored):
    row = scored[(run_name, seed)]
    budgets_here = sorted(row)
    accuracies = [row[b]["acc"] for b in budgets_here]
    rises = sum(1 for i in range(len(accuracies) - 1)
                if accuracies[i + 1] >= accuracies[i])
    flag = "" if rises >= len(accuracies) // 2 else "   <-- accuracy mostly FALLS with budget"
    print(f"{run_name[:43]:<44}seed={seed:<5}"
          f"{accuracies[0] * 100:.2f} -> {accuracies[-1] * 100:.2f}"
          f"  (up at {rises}/{len(accuracies) - 1} steps){flag}")
